# DL-01 Normalization Comparison

- Module: 07 Deep Learning Systems
- Topic: comparing plain training, batch normalization, and layer normalization in PyTorch
- Estimated runtime: 6 to 10 minutes once `torch` and `matplotlib` are installed
- Prerequisites: initialization heuristics, mini-batch SGD, and Module 06 notes on normalization
- Expected outputs: training curves, activation-statistics summaries, and a comparison across batch sizes


## Why this notebook exists

The goal is to compare three MLP variants on the same synthetic task:

- a plain deep network;
- a network with batch normalization;
- a network with layer normalization.

The mathematical distinction is the normalization axis.
Batch norm standardizes each feature across examples in a mini-batch.
Layer norm standardizes the features within each example.

This notebook intentionally compares both a moderate batch size and a tiny batch size.
That setup makes the practical difference visible instead of leaving it abstract.


In [ ]:
from __future__ import annotations

import random
from dataclasses import dataclass

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 7
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")


## Batch norm and layer norm formulas

For one hidden activation matrix $H \in \mathbb{R}^{m \times d}$:

- Batch norm computes mean and variance down each feature column.
- Layer norm computes mean and variance across each feature row.

In symbols,

$$
\mu_j^{\mathrm{BN}} = \frac{1}{m}\sum_{i=1}^{m} H_{ij},
\qquad
\mu_i^{\mathrm{LN}} = \frac{1}{d}\sum_{j=1}^{d} H_{ij}.
$$

The same axis choice propagates into the backward pass.
Batch norm couples examples in the same mini-batch, while layer norm does not.


In [ ]:
def make_dataset(n_samples: int = 2048):
    x = torch.empty(n_samples, 2).uniform_(-2.5, 2.5)
    radius = torch.linalg.norm(x, dim=1)
    angle = torch.atan2(x[:, 1], x[:, 0])
    score = radius + 0.35 * torch.sin(3.0 * angle)
    y = (score > 1.55).long()
    x = x + 0.15 * torch.randn_like(x)
    return x, y


x, y = make_dataset()
permutation = torch.randperm(len(x))
train_idx = permutation[:1536]
test_idx = permutation[1536:]

train_dataset = TensorDataset(x[train_idx], y[train_idx])
test_dataset = TensorDataset(x[test_idx], y[test_idx])


class DeepMLP(nn.Module):
    def __init__(self, norm: str | None = None, width: int = 64, depth: int = 8):
        super().__init__()
        layers: list[nn.Module] = []
        in_features = 2
        for _ in range(depth):
            layers.append(nn.Linear(in_features, width))
            if norm == "batch":
                layers.append(nn.BatchNorm1d(width))
            elif norm == "layer":
                layers.append(nn.LayerNorm(width))
            layers.append(nn.ReLU())
            in_features = width
        layers.append(nn.Linear(width, 2))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


@dataclass
class RunHistory:
    train_loss: list[float]
    test_loss: list[float]
    train_acc: list[float]
    test_acc: list[float]
    grad_norm: list[float]
    activation_std: list[float]


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(device))
            loss = loss_fn(logits, yb.to(device))
            total_loss += loss.item() * len(xb)
            total_correct += (logits.argmax(dim=1) == yb.to(device)).sum().item()
            total_count += len(xb)
    return total_loss / total_count, total_correct / total_count


def train_model(norm: str | None, batch_size: int, epochs: int = 20, lr: float = 5e-3):
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=256)
    model = DeepMLP(norm=norm).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    history = RunHistory([], [], [], [], [], [])

    for _ in range(epochs):
        model.train()
        epoch_loss = 0.0
        epoch_correct = 0
        total = 0
        grad_norms = []
        activation_stds = []
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            grad_norms.append(
                torch.sqrt(
                    sum((p.grad.detach() ** 2).sum() for p in model.parameters() if p.grad is not None)
                ).item()
            )
            optimizer.step()

            epoch_loss += loss.item() * len(xb)
            epoch_correct += (logits.argmax(dim=1) == yb).sum().item()
            total += len(xb)

            with torch.no_grad():
                first_linear = model.net[0](xb)
                activation_stds.append(first_linear.std().item())

        train_loss = epoch_loss / total
        train_acc = epoch_correct / total
        test_loss, test_acc = evaluate(model, test_loader, loss_fn)

        history.train_loss.append(train_loss)
        history.test_loss.append(test_loss)
        history.train_acc.append(train_acc)
        history.test_acc.append(test_acc)
        history.grad_norm.append(sum(grad_norms) / len(grad_norms))
        history.activation_std.append(sum(activation_stds) / len(activation_stds))
    return model, history


In [ ]:
configs = [
    ("plain-bs128", None, 128),
    ("batchnorm-bs128", "batch", 128),
    ("layernorm-bs128", "layer", 128),
    ("plain-bs8", None, 8),
    ("batchnorm-bs8", "batch", 8),
    ("layernorm-bs8", "layer", 8),
]

results = {}
for label, norm, batch_size in configs:
    _, history = train_model(norm=norm, batch_size=batch_size)
    results[label] = history

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for label, history in results.items():
    axes[0].plot(history.test_loss, label=label)
    axes[1].plot(history.test_acc, label=label)
    axes[2].plot(history.grad_norm, label=label)

axes[0].set_title("Test loss")
axes[1].set_title("Test accuracy")
axes[2].set_title("Mean gradient norm")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

{label: round(history.test_acc[-1], 3) for label, history in results.items()}


## Interpretation

Typical outcomes:

- the plain network trains, but more erratically, especially at depth;
- batch norm often gives the strongest performance at larger batch size;
- layer norm is usually more robust when the batch size is very small;
- batch norm can become noisier at tiny batch sizes because the batch statistics themselves are noisy.

This is one of the cleanest places to see a training pathology and its fix.
The pathology is unstable deep optimization without normalization.
The fixes are batch norm or layer norm, with the best choice depending on the data layout and batch regime.


## Exercises

1. Increase the network depth from `8` to `16` and compare how each normalization strategy degrades.
2. Replace `ReLU` with `Tanh`. Which model becomes most sensitive to depth?
3. Add a plot of the hidden-activation mean and variance at several layers instead of only the first one.
4. Change the tiny batch size from `8` to `2`. Does batch norm become noticeably less reliable?
